# Solving the Blasius Equation with the Shooting Method


---

## 1. Problem Statement

When a uniform stream $U_\infty$ flows over a flat plate, a thin boundary layer forms near the surface. Using the similarity variable

$$\eta = y\sqrt{\frac{U_\infty}{\nu x}}, \qquad \frac{u}{U_\infty} = f'(\eta),$$

the boundary layer equations reduce to a single third-order ODE, the **Blasius equation**:

$$f''' + \tfrac{1}{2} f f'' = 0, \qquad 0 < \eta < \infty$$

with boundary conditions

$$f(0) = 0, \quad f'(0) = 0 \quad \text{(no slip, no suction)}, \qquad f'(\eta) \to 1 \text{ as } \eta \to \infty \quad \text{(free stream)}.$$

The catch is that this is a **boundary value problem**. Two conditions are given at the wall and one is given "at infinity". Runge–Kutta schemes need all three conditions at the starting point, so we can't integrate it directly. That's where the shooting method comes in.



In [10]:

import numpy as np
import matplotlib.pyplot as plt


## 2.Writing it as a First-Order System

Let $\mathbf{Y} = (Y_1, Y_2, Y_3)^T = (f,\ f',\ f'')^T$. Then

$$
\mathbf{Y}' = \mathbf{G}(\mathbf{Y}) =
\begin{pmatrix} Y_2 \\ Y_3 \\ -\tfrac{1}{2} Y_1 Y_3 \end{pmatrix}
$$

The system is autonomous (there's no explicit $\eta$ on the right-hand side), which keeps the code a little simpler.

For the stepping I'm using Kutta's third-order Runge–Kutta scheme:

$$
\begin{aligned}
k_1 &= G(Y_n) \\
k_2 &= G(Y_n + \tfrac{h}{2} k_1) \\
k_3 &= G(Y_n - h k_1 + 2h k_2) \\
Y_{n+1} &= Y_n + \tfrac{h}{6}(k_1 + 4k_2 + k_3)
\end{aligned}
$$

and later the classical RK4 scheme for comparison.

In [11]:

def G(Y):
    """
    Right-hand side of the Blasius system.
    Y = [f, f', f'']  ->  Y' = [f', f'', -0.5 * f * f'']
    """
    f, fp, fpp = Y
    return np.array([fp, fpp, -0.5 * f * fpp])


def rk3_step(Y, h):
    """One step of Kutta's third-order Runge-Kutta method."""
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y - h * k1 + 2 * h * k2)
    return Y + (h / 6) * (k1 + 4 * k2 + k3)


def rk4_step(Y, h):
    """One step of the classical fourth-order Runge-Kutta method."""
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y + 0.5 * h * k2)
    k4 = G(Y + h * k3)
    return Y + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)


def integrate(s, h, eta_inf, stepper):
    """
    March the IVP from eta = 0 to eta = eta_inf with a fixed step h.
    The missing initial condition f''(0) is set to the guess s.
    Returns the eta grid and the solution array Y (columns: f, f', f'').
    """
    n = int(round(eta_inf / h))            
    eta = np.linspace(0.0, n * h, n + 1)   
    Y = np.zeros((n + 1, 3))

    
    Y[0] = [0.0, 0.0, s]

    
    for i in range(n):
        Y[i + 1] = stepper(Y[i], h)

    return eta, Y

## 3.The Missing Initial Condition

At $\eta = 0$ we know $f(0)=0$ and $f'(0)=0$. What we **don't** know is $f''(0)$, which physically is the non-dimensional wall shear stress. That becomes our shooting parameter:

$$s = f''(0)$$

For any guess of $s$ we get a well-posed initial value problem. The job is to find the value of $s$ that makes $f'$ approach 1 far from the wall.

## 4.Choosing $\eta_\infty$ and Setting Up the Residual

We can't integrate to actual infinity, so we cut the domain off at a finite $\eta_\infty$ and apply the far-field condition there. The residual is

$$F(s;\eta_\infty) = f'(\eta_\infty; s) - 1$$

and we want $F(s) = 0$.

**Root finder:** I used the **secant method**. It doesn't need the derivative of $F$ (Newton would need extra sensitivity equations), and it converges much faster than bisection. The starting guesses are $s_0 = 0.1$ and $s_1 = 0.5$, and iteration stops when successive values differ by less than $10^{-12}$. That tolerance is far below the discretisation errors we'll measure, so it won't affect the convergence study.

In [12]:

def residual(s, h, eta_inf, stepper):
    """F(s) = f'(eta_inf; s) - 1. We want this to be zero."""
    _, Y = integrate(s, h, eta_inf, stepper)
    return Y[-1, 1] - 1.0


def secant(h, eta_inf, stepper, s0=0.1, s1=0.5, tol=1e-12, max_iter=50, verbose=False):
    """
    Find s = f''(0) such that F(s) = 0 using the secant method.
    Returns the converged s and the number of iterations taken.
    """
    F0 = residual(s0, h, eta_inf, stepper)
    F1 = residual(s1, h, eta_inf, stepper)

    for it in range(1, max_iter + 1):
        
        s2 = s1 - F1 * (s1 - s0) / (F1 - F0)

        if verbose:
            print(f"  iter {it:2d}:  s = {s2:.12f}   |ds| = {abs(s2 - s1):.2e}")

        
        if abs(s2 - s1) < tol:
            return s2, it

        # Shift the points along for the next iteration
        s0, F0 = s1, F1
        s1, F1 = s2, residual(s2, h, eta_inf, stepper)

    raise RuntimeError("Secant method did not converge")



print("Secant iterations (RK4, h = 0.1, eta_inf = 10):")
s, n_it = secant(0.1, 10.0, rk4_step, verbose=True)
print(f"\nConverged to s = {s:.10f} in {n_it} iterations")

Secant iterations (RK4, h = 0.1, eta_inf = 10):
  iter  1:  s = 0.354843623278   |ds| = 1.45e-01
  iter  2:  s = 0.330384782704   |ds| = 2.45e-02
  iter  3:  s = 0.332076148221   |ds| = 1.69e-03
  iter  4:  s = 0.332057361417   |ds| = 1.88e-05
  iter  5:  s = 0.332057345610   |ds| = 1.58e-08
  iter  6:  s = 0.332057345611   |ds| = 1.49e-13

Converged to s = 0.3320573456 in 6 iterations


In [14]:

S_LABEL = "f''(0)"   # column header for the printed tables

print(f"{'eta_inf':>8} | {S_LABEL:>14}")
print("-" * 26)

for eta_inf in [4, 6, 8, 10, 12, 15]:
    s, _ = secant(0.01, eta_inf, rk4_step)
    print(f"{eta_inf:8.0f} | {s:14.10f}")

 eta_inf |         f''(0)
--------------------------
       4 |   0.3527462535
       6 |   0.3325659312
       8 |   0.3320591917
      10 |   0.3320573372
      12 |   0.3320573362
      15 |   0.3320573362


**Observation:** at $\eta_\infty = 4$ the answer is clearly off (about 0.353) because we're forcing $f'=1$ while still inside the boundary layer. At $\eta_\infty = 8$ it's right to about five digits, and from $\eta_\infty = 10$ onwards the change is around $10^{-9}$, which is smaller than the RK errors we're about to study.

So I'm using **$\eta_\infty = 10$**. It's large enough, and since $10/0.4 = 25$, every step size in part (d) lands exactly on the end point.

In [15]:

ETA_INF = 10.0                         
STEPS   = [0.4, 0.2, 0.1, 0.05, 0.025] 
h_arr   = np.array(STEPS)               